In [1]:
import numpy as np
import pandas as pd
from econml.dml import CausalForestDML
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LassoCV
import random
random.seed(42)

In [2]:
data = pd.read_csv("../../data/ice_cream_sales.csv")
data.head()

,ID,Temperature,Is_Weekend,Hours_Open,Electricity_Usage,Ice_Cream_Sales
0,1,24.273285,False,10,92.118314,983
1,2,25.503474,False,9,83.917817,1018
2,3,24.370024,False,8,88.290617,951
3,4,24.377495,False,10,85.561865,1012
4,5,26.632614,False,8,94.404976,1010


In [3]:
# Define the dependent variable (outcome)
y = data['Ice_Cream_Sales'].values

# Define the treatment variable
T = data['Temperature'].values

# Define the features (covariates)
X = data[['Electricity_Usage', 'Hours_Open', 'Is_Weekend']].values

In [4]:
X_train, X_test, T_train, T_test, y_train, y_test = train_test_split(
    X, T, y, test_size=0.2, random_state=42)

In [5]:
est = CausalForestDML(
    model_t=RandomForestRegressor(n_estimators=100, min_samples_leaf=5),
    model_y=RandomForestRegressor(n_estimators=100, min_samples_leaf=5),
    discrete_treatment=False,
    cv=3,
    n_estimators=1000,  # Increase for more accurate estimates
    min_samples_leaf=10,  # Increase to smooth the estimated treatment effects
    verbose=0,
    random_state=42
)

In [6]:
est.fit(y_train, T_train, X=X_train)

In [7]:
treatment_effects = est.effect(X_test)
ate = np.mean(treatment_effects)
print(ate)

28.754978925262193


In [14]:
ics_counterfactual = data["Ice_Cream_Sales"] + (30 - data["Temperature"]) * 30